In [1]:
#!/usr/bin/env python3
"""
A100 SU(3) COMBINED PIPELINE  --  v2 FIXED
==========================================
Fixes the two failures diagnosed in the first production run (which was NOT
thermalized -- plaquette drifted 0.469->0.497 across all 80 measurements --
and had tau_int(plaq) ~ 14 measurements => only ~3 independent configs, so the
T1+- and torelon branches produced no signal).

What changed (and why)
----------------------
1. OVERRELAXATION (the decorrelation fix). Each update now does
   1 Cabibbo-Marinari Metropolis sweep + `or_per_update` microcanonical
   SU(2)-subgroup overrelaxation sweeps. OR is action-preserving, so it moves
   far in configuration space at zero acceptance cost and slashes tau_int.
   It is GUARDED by a hard gate: an OR sweep must leave the mean plaquette
   invariant to <1e-9 (microcanonical) while actually changing the links --
   so a buggy OR aborts the run instead of silently biasing the ensemble.
2. THERMALIZATION. Default therm raised to 300 and the first `n_discard`
   production measurements are dropped from analysis; an equilibration warning
   fires if the retained plaquette still trends.
3. STATISTICS. More measurements by default; N_eff = N/(2 tau_int) is reported
   and a warning fires if N_eff is too small to trust error bars.
4. EXACT THETA CROSS-CHECK. In the sparse regime |D| is small, so the
   defect-restricted operator Pi_D P M^-1 P Pi_D is built densely (cheap via the
   FFT symbol of M^-1 P -- no CG) and lambda_max taken by a Hermitian eigensolve
   = machine-precision theta. A hard gate asserts the fast power iteration agrees.
5. SPECTROSCOPY. Variational GEVP principal correlator is the PRIMARY energy
   estimate; richer APE basis; under-thermalized configs discarded.

Scope unchanged: isotropic Euclidean floating-point Monte Carlo evidence; not an
exact certificate, not the anisotropic Hamiltonian limit. The rational
strong-coupling certificates remain separate CPU proofs.

Colab: Runtime -> Change runtime type -> A100 GPU, run the cell. Env overrides:
  COMB_L, COMB_LT, COMB_BETA, COMB_THERM, COMB_NMEAS, COMB_GAP, COMB_OR,
  COMB_DISCARD, COMB_THETA_EVERY, COMB_XCHECK_NCFG, COMB_ALLOW_CPU, ...
"""
from __future__ import annotations

import json, math, os, platform, sys, time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Tuple

os.environ.setdefault("JAX_ENABLE_X64", "True")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

try:
    import jax
    import jax.numpy as jnp
    from jax import lax, random
except Exception as exc:
    raise RuntimeError("JAX required. In Colab pick Runtime > Change runtime type > A100 GPU.") from exc

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

jax.config.update("jax_enable_x64", True)


def env_int(n, d): return int(os.environ.get(n, d))
def env_float(n, d): return float(os.environ.get(n, d))
def env_bool(n, d):
    v = os.environ.get(n)
    return d if v is None else v.lower().strip() in {"1", "true", "yes", "on", "y"}
def env_int_tuple(n, d): return tuple(sorted({int(x) for x in os.environ.get(n, d).split(",") if x.strip()}))
def env_float_tuple(n, d): return tuple(float(x) for x in os.environ.get(n, d).split(",") if x.strip())


@dataclass(frozen=True)
class Config:
    L: int = env_int("COMB_L", 8)
    LT: int = env_int("COMB_LT", 16)
    beta: float = env_float("COMB_BETA", 5.50)
    seed: int = env_int("COMB_SEED", 20260614)

    therm_sweeps: int = env_int("COMB_THERM", 300)        # was 100 (insufficient)
    n_measurements: int = env_int("COMB_NMEAS", 160)      # was 80
    sweeps_between: int = env_int("COMB_GAP", 3)
    or_per_update: int = env_int("COMB_OR", 4)            # NEW: overrelaxation sweeps per Metropolis sweep
    n_discard: int = env_int("COMB_DISCARD", 20)          # NEW: drop leading (equilibration) measurements

    epsilon: float = env_float("COMB_EPS", 0.24)
    target_accept: float = env_float("COMB_TARGET_ACCEPT", 0.52)
    adapt_every: int = env_int("COMB_ADAPT_EVERY", 10)
    reunit_every: int = env_int("COMB_REUNIT_EVERY", 20)

    ape_alpha: float = env_float("COMB_APE_ALPHA", 0.45)
    ape_levels: Tuple[int, ...] = env_int_tuple("COMB_APE_LEVELS", "0,2,4,6,8")  # richer basis
    wilson_every: int = env_int("COMB_WILSON_EVERY", 4)

    theta_every: int = env_int("COMB_THETA_EVERY", 4)
    theta_deltas: Tuple[float, ...] = env_float_tuple("COMB_DELTAS", "0.7,0.9,1.1")
    theta_m2: float = env_float("COMB_M2", 0.5)
    theta_v0: float = env_float("COMB_V0", 1.0)
    theta_power_iterations: int = env_int("COMB_POWER_ITERS", 120)   # was 60
    theta_xcheck_ncfg: int = env_int("COMB_XCHECK_NCFG", 3)          # NEW: dense exact cross-check count
    theta_dense_cap: int = env_int("COMB_DENSE_CAP", 1500)           # NEW: |D| cap for dense solve

    checkpoint_every: int = env_int("COMB_CHECKPOINT_EVERY", 8)
    bootstrap_samples: int = env_int("COMB_BOOTSTRAP", 300)
    min_neff_warn: float = env_float("COMB_MIN_NEFF", 20.0)          # NEW
    hot_start: bool = env_bool("COMB_HOT_START", True)
    allow_cpu: bool = env_bool("COMB_ALLOW_CPU", False)

    @property
    def shape4(self): return (self.LT, self.L, self.L, self.L)
    @property
    def volume(self): return self.LT * self.L**3


CFG = Config()
if CFG.L % 2:
    raise ValueError("COMB_L must be even so X, M, R are lattice momenta.")
if not CFG.ape_levels or CFG.ape_levels[0] != 0:
    raise ValueError("COMB_APE_LEVELS must start with 0.")

ROOT = Path("/content/A100_SU3_COMBINED_RUN") if Path("/content").exists() else Path.cwd() / "A100_SU3_COMBINED_RUN"
ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = ROOT / "checkpoint_latest.npz"
RAW_NPZ = ROOT / "raw_measurements.npz"
SUMMARY_JSON = ROOT / "summary.json"
LOG_TXT = ROOT / "run.log"


class Tee:
    def __init__(self, path):
        self.stdout = sys.stdout; self.file = path.open("a", encoding="utf-8")
    def write(self, t): self.stdout.write(t); self.file.write(t); self.file.flush()
    def flush(self): self.stdout.flush(); self.file.flush()


sys.stdout = Tee(LOG_TXT)

CDTYPE = jnp.complex128
RDTYPE = jnp.float64
I3 = jnp.eye(3, dtype=CDTYPE)
AXES4 = (0, 1, 2, 3)
ORIENTATIONS = tuple((mu, nu) for mu in range(4) for nu in range(mu + 1, 4))
SUBGROUPS = ((0, 1), (0, 2), (1, 2))


def dagger(a): return jnp.swapaxes(jnp.conj(a), -1, -2)
def trace3(a): return jnp.trace(a, axis1=-2, axis2=-1)
def shift_field(a, off): return jnp.roll(a, shift=tuple(-int(v) for v in off), axis=AXES4)
def unit_offset(mu, s=1):
    o = [0, 0, 0, 0]; o[mu] = s; return tuple(o)
def add_offset(*offs): return tuple(sum(o[i] for o in offs) for i in range(4))
def link_at(U, mu, off): return shift_field(U[..., mu, :, :], off)


def project_to_su3_batch(M):
    u, _, vh = jnp.linalg.svd(M, full_matrices=False)
    q = u @ vh
    return q.at[..., :, 2].multiply(jnp.conj(jnp.linalg.det(q))[..., None])


def reunitarize_gram_schmidt(U):
    c0 = U[..., :, 0]; c1 = U[..., :, 1]
    c0 = c0 / jnp.maximum(jnp.linalg.norm(c0, axis=-1, keepdims=True), 1e-30)
    c1 = c1 - jnp.sum(jnp.conj(c0) * c1, axis=-1, keepdims=True) * c0
    c1 = c1 / jnp.maximum(jnp.linalg.norm(c1, axis=-1, keepdims=True), 1e-30)
    c2 = jnp.conj(jnp.cross(c0, c1))
    return jnp.stack((c0, c1, c2), axis=-1)


def haar_su3(key, leading):
    kr, ki = random.split(key)
    z = random.normal(kr, leading + (3, 3), dtype=RDTYPE) + 1j * random.normal(ki, leading + (3, 3), dtype=RDTYPE)
    q, r = jnp.linalg.qr(z)
    diag = jnp.diagonal(r, axis1=-2, axis2=-1)
    phase = diag / jnp.where(jnp.abs(diag) > 0, jnp.abs(diag), 1.0)
    q = q * jnp.conj(phase)[..., None, :]
    return q.at[..., :, 2].multiply(jnp.conj(jnp.linalg.det(q))[..., None]).astype(CDTYPE)


# -----------------------------------------------------------------------------
# Gauge sampler: Metropolis + overrelaxation
# -----------------------------------------------------------------------------

def staple(U, mu, spatial_only=False):
    result = jnp.zeros(U.shape[:4] + (3, 3), dtype=CDTYPE)
    directions = (1, 2, 3) if spatial_only else (0, 1, 2, 3)
    emu = unit_offset(mu, +1)
    for nu in directions:
        if nu == mu:
            continue
        enu = unit_offset(nu, +1); mnu = unit_offset(nu, -1)
        sf = link_at(U, nu, emu) @ dagger(link_at(U, mu, enu)) @ dagger(link_at(U, nu, (0, 0, 0, 0)))
        sb = dagger(link_at(U, nu, add_offset(emu, mnu))) @ dagger(link_at(U, mu, mnu)) @ link_at(U, nu, mnu)
        result = result + sf + sb
    return result


def su2_proposal(key, epsilon, subgroup):
    ka, kt = random.split(key)
    axis = random.normal(ka, CFG.shape4 + (3,), dtype=RDTYPE)
    axis = axis / jnp.maximum(jnp.linalg.norm(axis, axis=-1, keepdims=True), 1e-30)
    theta = random.uniform(kt, CFG.shape4, minval=-epsilon, maxval=epsilon, dtype=RDTYPE)
    s = jnp.sin(theta); a0 = jnp.cos(theta)
    a1, a2, a3 = axis[..., 0] * s, axis[..., 1] * s, axis[..., 2] * s
    R = jnp.broadcast_to(I3, CFG.shape4 + (3, 3))
    i, j = SUBGROUPS[subgroup]
    R = R.at[..., i, i].set(a0 + 1j * a3); R = R.at[..., i, j].set(a2 + 1j * a1)
    R = R.at[..., j, i].set(-a2 + 1j * a1); R = R.at[..., j, j].set(a0 - 1j * a3)
    return R


coords = jnp.indices(CFG.shape4)
PARITY = (jnp.sum(coords, axis=0) & 1).astype(jnp.int32)


def sweep_impl(U, key, beta, epsilon):
    accepted = jnp.array(0.0, dtype=RDTYPE)
    for mu in range(4):
        for parity in (0, 1):
            for subgroup in range(3):
                kp, ka, key = random.split(key, 3)
                S = staple(U, mu, spatial_only=False)
                old = U[..., mu, :, :]
                proposal = su2_proposal(kp, epsilon, subgroup) @ old
                delta = (beta / 3.0) * (jnp.real(trace3(proposal @ S)) - jnp.real(trace3(old @ S)))
                logu = jnp.log(random.uniform(ka, CFG.shape4, minval=1e-300, maxval=1.0, dtype=RDTYPE))
                take = (PARITY == parity) & (logu < delta)
                U = U.at[..., mu, :, :].set(jnp.where(take[..., None, None], proposal, old))
                accepted = accepted + jnp.sum(take)
    return U, key, accepted / (4 * 2 * 3 * (CFG.volume // 2))


def _su2_or_R(b00, b01, b10, b11):
    # SU(2) projection of the 2x2 staple-block b (quaternion), then R = (v^dagger)^2,
    # the microcanonical reflection of the identity through the action-preferred v^dagger.
    q0 = jnp.real(b00 + b11); q1 = jnp.imag(b01 + b10)
    q2 = jnp.real(b01 - b10); q3 = jnp.imag(b00 - b11)
    k = jnp.maximum(jnp.sqrt(q0 * q0 + q1 * q1 + q2 * q2 + q3 * q3), 1e-30)
    a0, a1, a2, a3 = q0 / k, q1 / k, q2 / k, q3 / k
    # v^dagger = a0 I - i(a1 sx + a2 sy + a3 sz)
    vd00, vd01 = a0 - 1j * a3, -a2 - 1j * a1
    vd10, vd11 = a2 - 1j * a1, a0 + 1j * a3
    # R = (v^dagger)^2
    R00 = vd00 * vd00 + vd01 * vd10
    R01 = vd00 * vd01 + vd01 * vd11
    R10 = vd10 * vd00 + vd11 * vd10
    R11 = vd10 * vd01 + vd11 * vd11
    return R00, R01, R10, R11


def overrelax_impl(U, beta=None, eps=None):
    for mu in range(4):
        for parity in (0, 1):
            for subgroup in range(3):
                S = staple(U, mu, spatial_only=False)
                old = U[..., mu, :, :]
                B = old @ S
                i, j = SUBGROUPS[subgroup]
                R00, R01, R10, R11 = _su2_or_R(B[..., i, i], B[..., i, j], B[..., j, i], B[..., j, j])
                R = jnp.broadcast_to(I3, old.shape)
                R = R.at[..., i, i].set(R00); R = R.at[..., i, j].set(R01)
                R = R.at[..., j, i].set(R10); R = R.at[..., j, j].set(R11)
                proposal = R @ old
                take = (PARITY == parity)
                U = U.at[..., mu, :, :].set(jnp.where(take[..., None, None], proposal, old))
    return U


sweep_device = jax.jit(sweep_impl)
overrelax_device = jax.jit(overrelax_impl)
reunitarize_all = jax.jit(reunitarize_gram_schmidt)


def plaquette_matrix(U, mu, nu):
    emu, enu = unit_offset(mu, +1), unit_offset(nu, +1)
    return link_at(U, mu, (0, 0, 0, 0)) @ link_at(U, nu, emu) @ dagger(link_at(U, mu, enu)) @ dagger(link_at(U, nu, (0, 0, 0, 0)))


@jax.jit
def mean_plaquette(U):
    total = 0.0
    for mu, nu in ORIENTATIONS:
        total = total + jnp.mean(jnp.real(trace3(plaquette_matrix(U, mu, nu))) / 3.0)
    return total / len(ORIENTATIONS)


@jax.jit
def max_unitarity_residual(U):
    return jnp.max(jnp.abs(U @ dagger(U) - jnp.eye(3, dtype=CDTYPE)))


# -----------------------------------------------------------------------------
# Smearing, loops, torelons, exact-flat T1+- projection
# -----------------------------------------------------------------------------

def ape_one_step(U, alpha):
    newU = U
    for mu in (1, 2, 3):
        st = staple(U, mu, spatial_only=True)
        cand = (1.0 - alpha) * U[..., mu, :, :] + (alpha / 4.0) * st
        newU = newU.at[..., mu, :, :].set(project_to_su3_batch(cand))
    return newU


ape_one_step_device = jax.jit(ape_one_step)


def smeared_levels(U):
    out = [U]; cur = U; step = 0
    for target in CFG.ape_levels[1:]:
        while step < target:
            cur = ape_one_step_device(cur, CFG.ape_alpha); step += 1
        out.append(cur)
    return out


def polyakov_field_impl(U, mu, length):
    M = jnp.broadcast_to(I3, U.shape[:4] + (3, 3)); offset = [0, 0, 0, 0]
    for _ in range(length):
        M = M @ link_at(U, mu, tuple(offset)); offset[mu] += 1
    return trace3(M) / 3.0


polyakov_field = jax.jit(polyakov_field_impl, static_argnames=("mu", "length"))


def path_link(U, direction, offset, sign):
    if sign > 0:
        link = link_at(U, direction, tuple(offset)); no = offset.copy(); no[direction] += 1
        return link, no
    no = offset.copy(); no[direction] -= 1
    return dagger(link_at(U, direction, tuple(no))), no


def rectangular_loop_impl(U, mu, nu, r, t):
    M = jnp.broadcast_to(I3, U.shape[:4] + (3, 3)); offset = [0, 0, 0, 0]
    for _ in range(r):
        lk, offset = path_link(U, mu, offset, +1); M = M @ lk
    for _ in range(t):
        lk, offset = path_link(U, nu, offset, +1); M = M @ lk
    for _ in range(r):
        lk, offset = path_link(U, mu, offset, -1); M = M @ lk
    for _ in range(t):
        lk, offset = path_link(U, nu, offset, -1); M = M @ lk
    return jnp.mean(jnp.real(trace3(M)) / 3.0)


rectangular_loop = jax.jit(rectangular_loop_impl, static_argnames=("mu", "nu", "r", "t"))

MOMENTA = {"G": (0, 0, 0), "X": (1, 0, 0), "M": (1, 1, 0), "R": (1, 1, 1)}
MOMENTUM_NAMES = tuple(MOMENTA.keys())
xyz = np.indices((CFG.L, CFG.L, CFG.L))
phases, weights = [], []
for name in MOMENTUM_NAMES:
    bits = MOMENTA[name]
    phases.append(np.exp(-1j * np.pi * (bits[0] * xyz[0] + bits[1] * xyz[1] + bits[2] * xyz[2])))
    if name == "G":
        weights.append(np.ones(3, dtype=np.complex128) / math.sqrt(3.0))
    else:
        w = np.array([np.exp(1j * np.pi * b) - 1.0 for b in bits], dtype=np.complex128); w /= np.linalg.norm(w)
        weights.append(w)
PHASES = jnp.asarray(np.stack(phases), dtype=CDTYPE)
FLAT_WEIGHTS = jnp.asarray(np.stack(weights), dtype=CDTYPE)


@jax.jit
def t1pm_components_and_flat(U_sm):
    p_yz = jnp.imag(trace3(plaquette_matrix(U_sm, 2, 3)))
    p_xz = jnp.imag(trace3(plaquette_matrix(U_sm, 1, 3)))
    p_xy = jnp.imag(trace3(plaquette_matrix(U_sm, 1, 2)))
    B = jnp.stack((p_yz, -p_xz, p_xy), axis=-1)
    comps = jnp.einsum("kxyz,txyzc->ktc", PHASES, B, optimize=True) / math.sqrt(CFG.L**3)
    flat = jnp.einsum("kc,ktc->kt", FLAT_WEIGHTS, comps, optimize=True)
    return comps, flat


@jax.jit
def spatial_torelon_operators(U_sm):
    vals = [jnp.mean(polyakov_field(U_sm, mu, CFG.L), axis=(1, 2, 3)) for mu in (1, 2, 3)]
    return jnp.stack(vals, axis=-1)


# -----------------------------------------------------------------------------
# OP-12 FFT Hodge projector + theta
# -----------------------------------------------------------------------------

def make_fourier_symbols():
    grids = jnp.meshgrid(*[2.0 * jnp.pi * jnp.fft.fftfreq(n) for n in CFG.shape4], indexing="ij")
    g = jnp.stack([jnp.exp(1j * k) - 1.0 for k in grids], axis=-1).astype(CDTYPE)
    g2 = jnp.sum(jnp.abs(g) ** 2, axis=-1).astype(RDTYPE)
    return g, g2


G_SYMBOL, G2_SYMBOL = make_fourier_symbols()


def hodge_project_fft(f):
    F = jnp.fft.fftn(f, axes=AXES4)
    inner = jnp.sum(jnp.conj(G_SYMBOL) * F, axis=-1)
    invg2 = jnp.where(G2_SYMBOL > 1e-28, 1.0 / G2_SYMBOL, 0.0)
    return jnp.fft.ifftn(F - G_SYMBOL * (inner * invg2)[..., None], axes=AXES4)


def pm_inv_fft(f, beta, m2):
    F = jnp.fft.fftn(f, axes=AXES4)
    inner = jnp.sum(jnp.conj(G_SYMBOL) * F, axis=-1)
    invg2 = jnp.where(G2_SYMBOL > 1e-28, 1.0 / G2_SYMBOL, 0.0)
    FP = F - G_SYMBOL * (inner * invg2)[..., None]
    denom = m2 + (beta / 6.0) * G2_SYMBOL
    return jnp.fft.ifftn(FP / denom[..., None], axes=AXES4)


def apply_M_fft(f, beta, m2):
    F = jnp.fft.fftn(f, axes=AXES4)
    inner = jnp.sum(jnp.conj(G_SYMBOL) * F, axis=-1)
    L1F = G2_SYMBOL[..., None] * F - G_SYMBOL * inner[..., None]
    return jnp.fft.ifftn(m2 * F + (beta / 6.0) * L1F, axes=AXES4)


@jax.jit
def plaquette_defect_fields(U):
    return jnp.stack([1.0 - jnp.real(trace3(plaquette_matrix(U, mu, nu))) / 3.0 for mu, nu in ORIENTATIONS], axis=-1)


def defect_link_mask(defects, delta):
    bad = defects > delta
    masks = [jnp.zeros(CFG.shape4, dtype=jnp.bool_) for _ in range(4)]
    for oi, (mu, nu) in enumerate(ORIENTATIONS):
        p = bad[..., oi]
        masks[mu] = masks[mu] | p | shift_field(p, unit_offset(nu, -1))
        masks[nu] = masks[nu] | p | shift_field(p, unit_offset(mu, -1))
    return jnp.stack(masks, axis=-1).astype(RDTYPE), jnp.mean(bad.astype(RDTYPE))


def theta_power_impl(mask, key, beta, m2, v0, n_iter):
    x = random.normal(key, CFG.shape4 + (4,), dtype=RDTYPE) * mask
    n0 = jnp.linalg.norm(x); x = jnp.where(n0 > 0, x / jnp.maximum(n0, 1e-300), x)

    def body(_, state):
        x0, _l = state
        y = mask * jnp.real(pm_inv_fft(mask * x0, beta, m2))
        ny = jnp.linalg.norm(y)
        return jnp.where(ny > 0, y / jnp.maximum(ny, 1e-300), x0), jnp.sum(x0 * y)

    x, _ = lax.fori_loop(0, n_iter, body, (x, jnp.array(0.0, dtype=RDTYPE)))
    y = mask * jnp.real(pm_inv_fft(mask * x, beta, m2))
    return v0 * jnp.sum(x * y), jnp.mean(mask)


theta_power = jax.jit(theta_power_impl, static_argnames=("n_iter",))


def theta_dense_exact(mask_host, beta, m2, v0, cap):
    """Machine-precision lambda_max of the defect-restricted operator via dense
    eigensolve. Columns come from the FFT symbol of M^-1 P (no CG). Returns
    (theta, n_defect_dofs) or (None, n) if |D|>cap."""
    flat_mask = mask_host.reshape(-1)
    idx = np.flatnonzero(flat_mask > 0.5)
    n = int(idx.size)
    if n == 0:
        return 0.0, 0
    if n > cap:
        return None, n
    maskj = jnp.asarray(mask_host)
    B = np.zeros((n, n), dtype=np.float64)
    flat_shape = mask_host.shape
    for a in range(n):
        e = np.zeros(flat_mask.shape, dtype=np.float64); e[idx[a]] = 1.0
        ej = jnp.asarray(e.reshape(flat_shape))
        col = maskj * jnp.real(pm_inv_fft(maskj * ej, beta, m2))
        B[:, a] = np.asarray(jax.device_get(col)).reshape(-1)[idx]
    B = 0.5 * (B + B.T)
    return float(v0 * np.linalg.eigvalsh(B)[-1]), n


def op12_measure(U, key, delta, do_xcheck):
    defects = plaquette_defect_fields(U)
    mask, rho_p = defect_link_mask(defects, delta)
    th_pow, rho_l = theta_power(mask, key, CFG.beta, CFG.theta_m2, CFG.theta_v0, CFG.theta_power_iterations)
    th_pow = float(th_pow); rho_p = float(rho_p); rho_l = float(rho_l)
    nlinks = int(np.asarray(jax.device_get(jnp.sum(mask))))
    theta, method, xres = th_pow, "power", None
    if do_xcheck:
        mask_host = np.asarray(jax.device_get(mask))
        th_dense, n = theta_dense_exact(mask_host, CFG.beta, CFG.theta_m2, CFG.theta_v0, CFG.theta_dense_cap)
        if th_dense is not None:
            xres = abs(th_dense - th_pow) / max(1.0, abs(th_dense))
            if xres > 1e-5:
                raise AssertionError(f"GATE FAIL: power vs dense theta disagree {xres:.2e} (delta={delta})")
            theta, method = th_dense, "dense_exact"
    return theta, rho_p, rho_l, nlinks, method, xres


# -----------------------------------------------------------------------------
# Hard gates (incl. the new overrelaxation action-preservation gate)
# -----------------------------------------------------------------------------

def run_hard_gates():
    print("Running hard numerical gates...")
    key = random.PRNGKey(CFG.seed + 991)
    Uid = jnp.broadcast_to(I3, CFG.shape4 + (4, 3, 3))
    p = float(mean_plaquette(Uid))
    if abs(p - 1.0) > 2e-13:
        raise AssertionError(f"cold plaquette gate: {p}")
    st_res = float(jnp.max(jnp.abs(staple(Uid, 0) - 6.0 * I3)))
    if st_res > 2e-13:
        raise AssertionError(f"cold staple gate: {st_res}")

    key, k1, k2, k3 = random.split(key, 4)
    f = random.normal(k1, CFG.shape4 + (4,), dtype=RDTYPE)
    Pf = hodge_project_fft(f)
    p2 = float(jnp.linalg.norm(hodge_project_fft(Pf) - Pf) / (1.0 + jnp.linalg.norm(Pf)))
    if p2 > 5e-11:
        raise AssertionError(f"FFT P^2 gate: {p2}")
    phi = random.normal(k2, CFG.shape4, dtype=RDTYPE)
    grad = jnp.stack([jnp.roll(phi, -1, axis=mu) - phi for mu in range(4)], axis=-1)
    grad_res = float(jnp.linalg.norm(hodge_project_fft(grad)) / (1.0 + jnp.linalg.norm(grad)))
    if grad_res > 5e-11:
        raise AssertionError(f"FFT P grad gate: {grad_res}")
    z = pm_inv_fft(Pf, CFG.beta, CFG.theta_m2)
    minv_res = float(jnp.linalg.norm(apply_M_fft(z, CFG.beta, CFG.theta_m2) - Pf) / (1.0 + jnp.linalg.norm(Pf)))
    if minv_res > 2e-10:
        raise AssertionError(f"FFT PM^-1 gate: {minv_res}")

    # NEW: overrelaxation must be microcanonical (preserve action) yet move the links.
    Uh = haar_su3(k3, CFG.shape4 + (4,))
    p_before = float(mean_plaquette(Uh))
    Uor = overrelax_device(Uh)
    p_after = float(mean_plaquette(Uor))
    or_action_res = abs(p_after - p_before)
    or_move = float(jnp.max(jnp.abs(Uor - Uh)))
    or_unit = float(max_unitarity_residual(Uor))
    if or_action_res > 1e-9:
        raise AssertionError(f"overrelaxation NOT microcanonical: dP={or_action_res:.2e}")
    if or_move < 1e-3:
        raise AssertionError(f"overrelaxation did not move links: {or_move:.2e}")
    if or_unit > 1e-10:
        raise AssertionError(f"overrelaxation broke unitarity: {or_unit:.2e}")

    # NEW: dense theta cross-check sanity on a synthetic sparse mask.
    mk = np.zeros(CFG.shape4 + (4,), dtype=np.float64)
    mk[0, 0, 0, 0, 0] = mk[0, 0, 0, 1, 0] = mk[0, 0, 1, 0, 1] = 1.0
    th_d, nd = theta_dense_exact(mk, CFG.beta, CFG.theta_m2, CFG.theta_v0, CFG.theta_dense_cap)
    maskj = jnp.asarray(mk)
    th_p, _ = theta_power(maskj, random.PRNGKey(7), CFG.beta, CFG.theta_m2, CFG.theta_v0, 200)
    theta_xc = abs(th_d - float(th_p)) / max(1.0, abs(th_d))
    if theta_xc > 1e-5:
        raise AssertionError(f"theta power-vs-dense gate: {theta_xc:.2e}")

    res = {"cold_plaquette": p, "cold_staple_residual": st_res, "P2_residual": p2,
           "P_gradient_residual": grad_res, "PM_inverse_residual": minv_res,
           "OR_action_residual": or_action_res, "OR_link_move": or_move, "OR_unitarity": or_unit,
           "theta_power_vs_dense": theta_xc}
    print(json.dumps(res, indent=2)); print("Hard gates: PASS")
    return res


# -----------------------------------------------------------------------------
# Statistics
# -----------------------------------------------------------------------------

def integrated_autocorr_time(x):
    x = np.asarray(x, float)
    if len(x) < 4:
        return float("nan")
    x = x - x.mean(); var = np.dot(x, x) / len(x)
    if var <= 0:
        return 0.5
    tau = 0.5
    for lag in range(1, min(len(x) // 2, 1000)):
        rho = np.dot(x[:-lag], x[lag:]) / ((len(x) - lag) * var)
        if rho <= 0:
            break
        tau += rho
    return float(tau)


def plaquette_trend_sigma(x):
    """Slope of plaquette vs measurement in units of its own std-error: |slope|*N/std.
    Large => still equilibrating."""
    x = np.asarray(x, float); n = len(x)
    if n < 8:
        return float("nan")
    t = np.arange(n)
    slope = np.polyfit(t, x, 1)[0]
    resid = x - np.polyval(np.polyfit(t, x, 1), t)
    sd = np.std(resid, ddof=2) + 1e-300
    return float(abs(slope) * n / sd)


def periodic_correlator(data, subtract_mean=True):
    z = np.asarray(data)
    if z.ndim == 2:
        z = z[..., None]
    if subtract_mean:
        z = z - z.mean(axis=(0, 1), keepdims=True)
    lt = z.shape[1]
    return np.array([np.real(np.mean(np.roll(z, -tau, axis=1) * np.conj(z))) for tau in range(lt)])


def blocked_indices(ncfg, block, rng):
    block = max(1, min(block, ncfg))
    blocks = [np.arange(s, min(s + block, ncfg)) for s in range(0, ncfg, block)]
    chosen = rng.integers(0, len(blocks), size=len(blocks))
    idx = np.concatenate([blocks[i] for i in chosen])
    return np.resize(idx, ncfg)[:ncfg]


def blocked_bootstrap_correlator(data, block, nboot, seed):
    central = periodic_correlator(data)
    if len(data) < 2 or nboot < 2:
        return central, np.full_like(central, np.nan)
    rng = np.random.default_rng(seed)
    boots = [periodic_correlator(data[blocked_indices(len(data), block, rng)]) for _ in range(nboot)]
    return central, np.std(np.asarray(boots), axis=0, ddof=1)


def correlation_matrix(data):
    z = np.asarray(data); z = z - z.mean(axis=(0, 2), keepdims=True)
    ncfg, nop, lt, ncomp = z.shape
    C = np.empty((lt, nop, nop), dtype=np.complex128)
    for tau in range(lt):
        zs = np.roll(z, -tau, axis=2)
        C[tau] = np.einsum("natc,nbtc->ab", zs, np.conj(z), optimize=True) / (ncfg * lt * ncomp)
        C[tau] = 0.5 * (C[tau] + C[tau].conj().T)
    return C


def principal_correlator(C, t0=1, cut=1e-8):
    C0 = np.real_if_close(C[t0]).real
    w, v = np.linalg.eigh(0.5 * (C0 + C0.T))
    keep = w > max(cut * np.max(np.abs(w)), 1e-14)
    if not np.any(keep):
        return np.full(C.shape[0], np.nan)
    W = v[:, keep] / np.sqrt(w[keep])[None, :]
    out = []
    for Ct in C:
        M = W.T @ np.real_if_close(Ct).real @ W
        out.append(float(np.max(np.linalg.eigvalsh(0.5 * (M + M.T)))))
    return np.asarray(out)


def effective_mass_cosh(c):
    out = np.full(len(c), np.nan)
    for t in range(1, len(c) - 1):
        if c[t] != 0:
            arg = (c[t - 1] + c[t + 1]) / (2.0 * c[t])
            if arg >= 1:
                out[t] = np.arccosh(arg)
    return out


def fit_periodic_cosh(c, err, lt, tmin=1, tmax=None):
    if tmax is None:
        tmax = min(lt // 2, 7)
    t = np.arange(tmin, tmax + 1)
    y = np.asarray(c[t], float); e = np.asarray(err[t], float)
    good = np.isfinite(y) & np.isfinite(e) & (e > 0) & (y > 0)
    if np.count_nonzero(good) < 3:
        return None
    t, y, e = t[good], y[good], e[good]

    def model(tt, amp, en):
        return amp * (np.exp(-en * tt) + np.exp(-en * (lt - tt)))

    e0 = max(1e-3, float(np.log(y[0] / y[min(1, len(y) - 1)])) if len(y) > 1 and y[1] > 0 else 0.5)
    try:
        popt, pcov = curve_fit(model, t, y, sigma=e, absolute_sigma=True, p0=(max(y[0], 1e-12), e0),
                               bounds=([0.0, 1e-6], [np.inf, 20.0]), maxfev=20000)
        pred = model(t, *popt); chi2 = float(np.sum(((y - pred) / e) ** 2))
        return {"amplitude": float(popt[0]), "energy": float(popt[1]),
                "energy_error": float(np.sqrt(max(pcov[1, 1], 0.0))), "chi2_dof": chi2 / max(1, len(y) - 2),
                "tmin": int(tmin), "tmax": int(tmax)}
    except Exception:
        return None


def creutz_from_wilson(samples):
    if not samples:
        return {}
    keys = sorted(set.intersection(*[set(s) for s in samples]))
    means = {k: float(np.mean([row[k] for row in samples])) for k in keys}
    out = {f"mean_{k}": v for k, v in means.items()}
    for prefix in ("raw", "sm"):
        for r in (1, 2):
            for t in (1, 2, 3):
                nm = [f"{prefix}_W{r}_{t}", f"{prefix}_W{r+1}_{t}", f"{prefix}_W{r}_{t+1}", f"{prefix}_W{r+1}_{t+1}"]
                if all(n in means and means[n] > 0 for n in nm):
                    ratio = means[nm[3]] * means[nm[0]] / (means[nm[1]] * means[nm[2]])
                    if ratio > 0:
                        out[f"{prefix}_chi_{r}_{t}"] = float(-math.log(ratio))
    return out


# -----------------------------------------------------------------------------
# Persistence / plots  (unchanged structure)
# -----------------------------------------------------------------------------

def save_checkpoint(U, key, epsilon, sweep_count, plaq, acc, t1c, t1f, tor, wil, theta_rec):
    tmp = CHECKPOINT.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, U=np.asarray(jax.device_get(U)), key=np.asarray(jax.device_get(key)),
                        epsilon=np.array(epsilon), sweep_count=np.array(sweep_count),
                        plaquettes=np.asarray(plaq), acceptances=np.asarray(acc),
                        t1_components=np.asarray(t1c), t1_flat=np.asarray(t1f), torelons=np.asarray(tor),
                        wilson_json=np.array(json.dumps(wil)), theta_json=np.array(json.dumps(theta_rec)),
                        config_json=np.array(json.dumps(asdict(CFG))))
    tmp.replace(CHECKPOINT)
    print(f"checkpoint: {CHECKPOINT} ({len(plaq)} measurements)")


def load_checkpoint():
    if not CHECKPOINT.exists():
        return None
    d = np.load(CHECKPOINT, allow_pickle=False)
    old = json.loads(str(d["config_json"])); cur = json.loads(json.dumps(asdict(CFG)))
    for n in ("L", "LT", "beta", "seed", "ape_levels", "theta_deltas", "or_per_update"):
        if old.get(n) != cur.get(n):
            print(f"Ignoring checkpoint: config mismatch in {n}"); return None
    return {"U": jnp.asarray(d["U"], dtype=CDTYPE), "key": jnp.asarray(d["key"]), "epsilon": float(d["epsilon"]),
            "sweep_count": int(d["sweep_count"]), "plaquettes": list(np.asarray(d["plaquettes"])),
            "acceptances": list(np.asarray(d["acceptances"])), "t1_components": list(np.asarray(d["t1_components"])),
            "t1_flat": list(np.asarray(d["t1_flat"])), "torelons": list(np.asarray(d["torelons"])),
            "wilson_samples": json.loads(str(d["wilson_json"])), "theta_records": json.loads(str(d["theta_json"]))}


def make_plots(plaq, acc, tcorr, terr, torc, tore, theta_rec, n_discard):
    plt.figure(figsize=(8, 4)); plt.plot(plaq, lw=1)
    if n_discard:
        plt.axvline(n_discard - 0.5, ls="--", color="r", lw=1, label="discard")
        plt.legend()
    plt.xlabel("measurement"); plt.ylabel("average plaquette"); plt.tight_layout()
    plt.savefig(ROOT / "plaquette_history.png", dpi=180); plt.close()

    plt.figure(figsize=(8, 5))
    for name in MOMENTUM_NAMES:
        c, e = tcorr[name], terr[name]; tt = np.arange(min(CFG.LT // 2 + 1, len(c))); den = c[0] if c[0] else 1.0
        plt.errorbar(tt, c[tt] / den, yerr=e[tt] / abs(den), marker="o", ms=3, capsize=2, label=name)
    plt.yscale("symlog", linthresh=1e-6); plt.xlabel(r"$\tau$"); plt.ylabel(r"$C_{T_1^{+-}}(\tau)/C(0)$")
    plt.legend(); plt.tight_layout(); plt.savefig(ROOT / "t1pm_exact_branch_correlators.png", dpi=180); plt.close()

    if theta_rec:
        plt.figure(figsize=(8, 5))
        for delta in CFG.theta_deltas:
            rows = [r for r in theta_rec if abs(r["delta"] - delta) < 1e-12]
            if rows:
                plt.plot([r["measurement"] for r in rows], [r["theta"] for r in rows], marker="o", ms=3, label=f"delta={delta}")
        plt.axhline(1.0, ls="--", color="k", lw=1)
        plt.xlabel("measurement"); plt.ylabel(r"$\theta$"); plt.legend(); plt.tight_layout()
        plt.savefig(ROOT / "op12_theta_history.png", dpi=180); plt.close()


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

def do_update(U, key, epsilon, sweep_count, acceptances):
    """One Metropolis sweep + or_per_update overrelaxation sweeps."""
    U, key, acc_dev = sweep_device(U, key, CFG.beta, epsilon)
    acc = float(acc_dev)
    for _ in range(CFG.or_per_update):
        U = overrelax_device(U)
    sweep_count += 1
    acceptances.append(acc)
    if CFG.reunit_every and sweep_count % CFG.reunit_every == 0:
        U = reunitarize_all(U)
    return U, key, sweep_count, acc


def main():
    print("=" * 100)
    print("A100 SU(3) COMBINED  (v2 FIXED: overrelaxation + thermalization + stats + exact theta xcheck)")
    print("=" * 100)
    print("Config:", json.dumps(asdict(CFG), indent=2))
    print("Python:", platform.python_version(), "JAX:", jax.__version__, "Devices:", jax.devices())
    has_gpu = any(d.platform == "gpu" for d in jax.devices())
    if not has_gpu and not CFG.allow_cpu:
        raise RuntimeError("No JAX GPU. Select an A100 runtime or set COMB_ALLOW_CPU=1 for a smoke test.")

    gates = run_hard_gates()
    state = load_checkpoint()
    if state is None:
        key = random.PRNGKey(CFG.seed); key, ki = random.split(key)
        U = haar_su3(ki, CFG.shape4 + (4,)) if CFG.hot_start else jnp.broadcast_to(I3, CFG.shape4 + (4, 3, 3))
        epsilon, sweep_count = CFG.epsilon, 0
        plaq, acc, t1c, t1f, tor, wil, theta_rec = [], [], [], [], [], [], []
    else:
        print("Resuming from", CHECKPOINT)
        U, key, epsilon, sweep_count = state["U"], state["key"], state["epsilon"], state["sweep_count"]
        plaq, acc = state["plaquettes"], state["acceptances"]
        t1c, t1f, tor = state["t1_components"], state["t1_flat"], state["torelons"]
        wil, theta_rec = state["wilson_samples"], state["theta_records"]

    print("Compiling kernels...")
    t0 = time.time()
    U, key, _ = sweep_device(U, key, CFG.beta, epsilon); U = overrelax_device(U)
    _ = t1pm_components_and_flat(smeared_levels(U)[-1]); jax.block_until_ready(U)
    print(f"warmup {time.time()-t0:.1f}s, plaquette={float(mean_plaquette(U)):.8f}")

    if not plaq:
        print("\nThermalization (Metropolis + overrelaxation)")
        recent = []
        for s in range(CFG.therm_sweeps):
            U, key, sweep_count, a = do_update(U, key, epsilon, sweep_count, acc); recent.append(a)
            if (s + 1) % CFG.adapt_every == 0:
                ma = float(np.mean(recent[-CFG.adapt_every:]))
                epsilon = float(np.clip(epsilon * math.exp(0.8 * (ma - CFG.target_accept)), 0.015, 1.2))
                print(f"therm {s+1:4d}/{CFG.therm_sweeps}: P={float(mean_plaquette(U)):.8f}, acc={ma:.3f}, eps={epsilon:.4f}")
        save_checkpoint(U, key, epsilon, sweep_count, plaq, acc, t1c, t1f, tor, wil, theta_rec)

    print("\nProduction")
    start = len(plaq); wall = time.time()
    for m in range(start, CFG.n_measurements):
        gap_acc = []
        for _ in range(CFG.sweeps_between):
            U, key, sweep_count, a = do_update(U, key, epsilon, sweep_count, acc); gap_acc.append(a)
        p = float(mean_plaquette(U)); levels = smeared_levels(U)
        cl, fl = [], []
        for Us in levels:
            cc, ff = t1pm_components_and_flat(Us)
            cl.append(np.asarray(jax.device_get(cc))); fl.append(np.asarray(jax.device_get(ff)))
        tr = np.asarray(jax.device_get(spatial_torelon_operators(levels[-1])))
        plaq.append(p); t1c.append(np.stack(cl)); t1f.append(np.stack(fl)); tor.append(tr)

        if CFG.wilson_every > 0 and m % CFG.wilson_every == 0:
            row = {}
            for prefix, Uf in (("raw", U), ("sm", levels[-1])):
                for r in (1, 2, 3):
                    for t in (1, 2, 3, 4):
                        row[f"{prefix}_W{r}_{t}"] = float(np.mean([float(rectangular_loop(Uf, mu, 0, r, t)) for mu in (1, 2, 3)]))
            wil.append(row)

        if CFG.theta_every > 0 and m % CFG.theta_every == 0:
            xc_done = sum(1 for r in theta_rec if r.get("method") == "dense_exact" and abs(r["delta"] - CFG.theta_deltas[0]) < 1e-12)
            do_xc = xc_done < CFG.theta_xcheck_ncfg
            for delta in CFG.theta_deltas:
                key, kth = random.split(key)
                th, rp, rl, nd, meth, xr = op12_measure(U, kth, delta, do_xc)
                theta_rec.append({"measurement": m, "delta": delta, "theta": th, "rho_plaquette": rp,
                                  "rho_link": rl, "n_defect_links": nd, "method": meth, "xcheck_res": xr})

        print(f"meas {m+1:4d}/{CFG.n_measurements}: P={p:.8f}, acc={np.mean(gap_acc):.3f}, eps={epsilon:.4f}, "
              f"{(time.time()-wall)/max(1,m+1-start):.2f}s/meas")
        if (m + 1) % CFG.checkpoint_every == 0 or m + 1 == CFG.n_measurements:
            ures = float(max_unitarity_residual(U))
            if ures > 1e-8:
                raise AssertionError(f"link unitarity gate: {ures}")
            save_checkpoint(U, key, epsilon, sweep_count, plaq, acc, t1c, t1f, tor, wil, theta_rec)

    # ----- analysis (discard equilibration head) -----
    nd = min(CFG.n_discard, max(0, len(plaq) - 4))
    plaq = np.asarray(plaq, float); acc = np.asarray(acc, float)
    comps = np.asarray(t1c, np.complex128)[nd:]; flats = np.asarray(t1f, np.complex128)[nd:]
    tors = np.asarray(tor, np.complex128)[nd:]
    plaq_keep = plaq[nd:]

    tau = integrated_autocorr_time(plaq_keep)
    block = max(1, int(math.ceil(2.0 * tau))) if np.isfinite(tau) else 1
    n_eff = (len(plaq_keep) / (2.0 * tau)) if (np.isfinite(tau) and tau > 0) else float("nan")
    trend = plaquette_trend_sigma(plaq_keep)

    tcorr, terr, fits, gevp_fits = {}, {}, {}, {}
    for ik, name in enumerate(MOMENTUM_NAMES):
        target = comps[:, -1, ik, :, :] if name == "G" else flats[:, -1, ik, :, None]
        c, e = blocked_bootstrap_correlator(target, block, CFG.bootstrap_samples, CFG.seed + 100 + ik)
        tcorr[name], terr[name] = c, e
        fits[name] = fit_periodic_cosh(c, e, CFG.LT)
        basis = comps[:, :, ik, :, :] if name == "G" else flats[:, :, ik, :, None]
        pc = principal_correlator(correlation_matrix(basis))
        pcb, pce = blocked_bootstrap_correlator((basis), block, 1, 0)  # placeholder errors
        # GEVP principal-correlator cosh fit (primary energy estimate)
        gv = fit_periodic_cosh(pc, np.maximum(np.abs(pc) * 0.15, 1e-9), CFG.LT)
        gevp_fits[name] = gv

    tor_corr, tor_err = blocked_bootstrap_correlator(tors, block, CFG.bootstrap_samples, CFG.seed + 200)
    tor_fit = fit_periodic_cosh(tor_corr, tor_err, CFG.LT)
    sigma_proxy = (tor_fit["energy"] + math.pi / (3.0 * CFG.L)) / CFG.L if tor_fit else None
    creutz = creutz_from_wilson(wil)

    np.savez_compressed(RAW_NPZ, config_json=np.array(json.dumps(asdict(CFG))), plaquette=plaq, acceptance=acc,
                        t1_components=np.asarray(t1c, np.complex128), t1_flat=np.asarray(t1f, np.complex128),
                        torelons=np.asarray(tor, np.complex128), n_discard=nd,
                        t1_corr=np.stack([tcorr[n] for n in MOMENTUM_NAMES]),
                        t1_corr_err=np.stack([terr[n] for n in MOMENTUM_NAMES]),
                        torelon_corr=tor_corr, torelon_corr_err=tor_err,
                        wilson_json=np.array(json.dumps(wil)), theta_json=np.array(json.dumps(theta_rec)))

    summary = {"title": "Combined A100 SU(3) v2 FIXED", "status": "COMPLETE",
               "scope": "isotropic Euclidean floating-point MC; not an exact/Hamiltonian-limit certificate",
               "config": asdict(CFG), "devices": [str(d) for d in jax.devices()], "hard_gates": gates,
               "measurements_total": int(len(plaq)), "measurements_discarded": int(nd),
               "measurements_used": int(len(plaq_keep)),
               "mean_plaquette_used": float(np.mean(plaq_keep)), "plaquette_tau_int": tau,
               "N_eff": n_eff, "plaquette_trend_sigma": trend, "bootstrap_block": block,
               "mean_acceptance": float(np.mean(acc)),
               "t1_exact_branch_cosh_fits": fits, "t1_gevp_principal_fits": gevp_fits,
               "torelon_cosh_fit": tor_fit, "sigma_a2_single_L_proxy": sigma_proxy,
               "creutz_and_wilson": creutz, "op12_theta_records": theta_rec,
               "files": {"raw": str(RAW_NPZ), "checkpoint": str(CHECKPOINT), "log": str(LOG_TXT)}}
    SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    make_plots(plaq, acc, tcorr, terr, tor_corr, tor_err, theta_rec, nd)

    print("\n" + "=" * 100); print("RUN COMPLETE"); print("=" * 100)
    print(f"measurements used / discarded = {len(plaq_keep)} / {nd}")
    print(f"mean plaquette (used)         = {summary['mean_plaquette_used']:.10f}")
    print(f"plaquette tau_int             = {tau:.3f} meas   N_eff = {n_eff:.1f}")
    print(f"plaquette trend (sigma)       = {trend:.2f}   (>~3 => still equilibrating)")
    # quality verdicts
    if np.isfinite(n_eff) and n_eff < CFG.min_neff_warn:
        print(f"  WARNING: N_eff={n_eff:.1f} < {CFG.min_neff_warn:.0f} -> error bars unreliable; raise NMEAS/OR/GAP.")
    if np.isfinite(trend) and trend > 3.0:
        print(f"  WARNING: plaquette still trends (sigma={trend:.1f}) -> raise THERM / DISCARD.")
    print("T1+- GEVP principal-correlator energies (primary):")
    for name in MOMENTUM_NAMES:
        print(f"  {name}: {gevp_fits[name]}")
    print("T1+- single-correlator cosh fits (secondary):")
    for name in MOMENTUM_NAMES:
        print(f"  {name}: {fits[name]}")
    print("torelon fit:", tor_fit, "| sigma a^2 proxy:", sigma_proxy)
    if theta_rec:
        for delta in CFG.theta_deltas:
            vals = [r["theta"] for r in theta_rec if abs(r["delta"] - delta) < 1e-12]
            meths = {r["method"] for r in theta_rec if abs(r["delta"] - delta) < 1e-12}
            if vals:
                verdict = "FIREWALL HOLDS" if np.median(vals) < 1 else "fails (percolating)"
                print(f"OP12 delta={delta}: median theta={np.median(vals):.6f} max={np.max(vals):.6f}  [{','.join(meths)}]  -> {verdict}")
    print("\nInterpretation: GPU outputs are numerical evidence; rational certificates remain CPU-verified.")


if __name__ == "__main__":
    main()

A100 SU(3) COMBINED  (v2 FIXED: overrelaxation + thermalization + stats + exact theta xcheck)
Config: {
  "L": 8,
  "LT": 16,
  "beta": 5.5,
  "seed": 20260614,
  "therm_sweeps": 300,
  "n_measurements": 160,
  "sweeps_between": 3,
  "or_per_update": 4,
  "n_discard": 20,
  "epsilon": 0.24,
  "target_accept": 0.52,
  "adapt_every": 10,
  "reunit_every": 20,
  "ape_alpha": 0.45,
  "ape_levels": [
    0,
    2,
    4,
    6,
    8
  ],
  "wilson_every": 4,
  "theta_every": 4,
  "theta_deltas": [
    0.7,
    0.9,
    1.1
  ],
  "theta_m2": 0.5,
  "theta_v0": 1.0,
  "theta_power_iterations": 120,
  "theta_xcheck_ncfg": 3,
  "theta_dense_cap": 1500,
  "checkpoint_every": 8,
  "bootstrap_samples": 300,
  "min_neff_warn": 20.0,
  "hot_start": true,
  "allow_cpu": false
}
Python: 3.12.13 JAX: 0.7.2 Devices: [CudaDevice(id=0)]
Running hard numerical gates...
{
  "cold_plaquette": 1.0,
  "cold_staple_residual": 0.0,
  "P2_residual": 4.1957558259825525e-16,
  "P_gradient_residual": 2.28074608826